# Central stellar-to-halo mass relation as a probe of $\tau_0$

The dynamical-friction timescale $\tau_0$ controls how long a satellite spirals in before merging with its central. The most direct stellar-mass-budget consequence is on the **brightest central galaxy (BCG)** — the central of the dominant subhalo of each FOF group. Two limiting behaviours:

* $\tau_0 \to 0$: every infalling satellite merges with the BCG immediately, so the BCG accretes all the stellar mass that would otherwise live in satellites.
* $\tau_0 \to \infty$: satellites never merge, so the BCG is starved of accreted stellar mass and stays close to the mass it built in situ.

If the dynamical-friction timescale matters in the high-$M_\mathrm{halo}$ regime, the bracket between these two limits should be visible in the central SHMR — particularly above $\log_{10}(M_\mathrm{halo}/[M_\odot/h]) \sim 13$. This notebook quantifies that bracket using the GALFORM L800/lc16.newmg runs at $z=0$ and $z=0.5$.

## Method

* **FOF-central selection.** A galaxy counts as the BCG of its FOF group if `is_central == 1` AND `mhalo / mhhalo > 0.5` (own subhalo dominates the group). One BCG per FOF.
* **Per-subvolume statistic.** In each subvolume we bin BCGs by host halo mass and record the median $\log_{10}(M_\star)$ along with the 16/84-percentile band.
* **Stacking.** Per-subvolume statistics from 16 ivols are averaged. Bootstrap resampling over ivols gives a 16/84 range on the mean.
* **Ratio panels.** $M_\star^\mathrm{model}/M_\star^\mathrm{default}$ at fixed $M_\mathrm{halo}$ isolates the $\tau_0$ effect from the baseline cosmology.

All masses are in $M_\odot/h$.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from utils.matplotlib_config import setconfig

sys.path.insert(0, str(Path.cwd()))
from tau0_helpers import (
    DEFAULT_IVOLS,
    RUNS,
    RUN_LABELS,
    SNAPSHOTS,
    central_shmr_per_ivol,
    collect_per_ivol,
    fof_central_mask,
    load_galaxy_fields,
    safe_ratio,
    stack_per_ivol,
    style_for,
)

setconfig({
    'figure': {'figsize': (8, 6)},
    'font': {'size': 14},
    'legend': {'fontsize': 12, 'title_fontsize': 13},
    'axes': {'labelsize': 14},
})

halo_bins = np.arange(11.0, 15.51, 0.25)  # log10(Mhhalo / Msun h^-1)
halo_centers = 0.5 * (halo_bins[1:] + halo_bins[:-1])
ivols = DEFAULT_IVOLS

summarise_shmr = lambda d: central_shmr_per_ivol(d, halo_bins)

## Compute the per-ivol SHMR for every (run, snapshot)

Each `stacked[snapshot][run]` is the output of `stack_per_ivol`, holding the bin-centres and arrays of `mean`, `sem`, `boot_lo`, `boot_hi` for the `median`, `p16`, `p84`, and `counts` keys.

In [ ]:
stacked = {}
for snapshot in SNAPSHOTS:
    stacked[snapshot] = {}
    for run_label, run_path in RUNS.items():
        summaries = collect_per_ivol(run_path, snapshot, ivols, summarise_shmr)
        stacked[snapshot][run_label] = stack_per_ivol(
            summaries,
            keys=('median', 'p16', 'p84', 'counts'),
            nboot=500,
            seed=23,
        )
        print(
            f"{snapshot} {run_label:>10s}: "
            f"{stacked[snapshot][run_label]['n_used']:2d} ivols used"
        )

## Figure 1 — Central SHMR with bootstrap range and IQR scatter

Two columns (z=0, z=0.5). Top row: median $\log_{10}(M_\star)$ vs $\log_{10}(M_\mathrm{halo})$, bootstrap 16-84 band on the mean. Background dotted lines show the 16/84 *intrinsic* scatter band averaged across ivols. Bottom row: ratio of $M_\star$ to the Default run.

In [ ]:
fig, axes = plt.subplots(
    2,
    len(SNAPSHOTS),
    figsize=(11, 8),
    sharex='col',
    gridspec_kw={'height_ratios': [3, 1.5], 'hspace': 0.05, 'wspace': 0.05},
)
if len(SNAPSHOTS) == 1:
    axes = axes[:, None]

min_count = 30  # don't plot bins with less than this many BCGs (per-ivol mean)

for j, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    ax_main = axes[0, j]
    ax_ratio = axes[1, j]

    default_med = stacked[snapshot]['Default']['median']['mean']
    for run_label in RUNS:
        s = stacked[snapshot][run_label]
        if s['n_used'] == 0:
            continue
        median_mean = s['median']['mean']
        boot_lo = s['median']['boot_lo']
        boot_hi = s['median']['boot_hi']
        n_bin_mean = s['counts']['mean']

        ok = np.isfinite(median_mean) & (n_bin_mean >= min_count)
        st = style_for(run_label)

        ax_main.plot(halo_centers[ok], median_mean[ok], '-', lw=2.2, **st)
        ax_main.fill_between(
            halo_centers[ok], boot_lo[ok], boot_hi[ok],
            color=st['color'], alpha=0.20, linewidth=0,
        )

        # Intrinsic IQR (16/84 percentile of log Mstar within each Mhalo bin),
        # averaged over ivols. Only show for Default to avoid clutter.
        if run_label == 'Default':
            p16 = s['p16']['mean']
            p84 = s['p84']['mean']
            ax_main.plot(halo_centers[ok], p16[ok], ':', color=st['color'], lw=1, alpha=0.7)
            ax_main.plot(halo_centers[ok], p84[ok], ':', color=st['color'], lw=1, alpha=0.7,
                         label='Default 16/84 scatter')

        # Ratio panel: Mstar^model / Mstar^default at fixed Mhalo (in linear space)
        if run_label == 'Default':
            continue
        ratio = 10 ** (median_mean - default_med)
        ax_ratio.plot(halo_centers[ok], ratio[ok], '-', lw=2.2, **st)
        # Bootstrap uncertainty: convert delta-log range to linear ratio range
        ratio_lo = 10 ** (boot_lo - default_med)
        ratio_hi = 10 ** (boot_hi - default_med)
        ax_ratio.fill_between(
            halo_centers[ok], ratio_lo[ok], ratio_hi[ok],
            color=st['color'], alpha=0.20, linewidth=0,
        )

    ax_main.set_title(f"{snapshot}  (z = {z_val:.1f})")
    ax_main.set_ylabel(r'$\log_{10}\,M_{\star,\,\mathrm{cen}}\ [M_\odot/h]$')
    ax_main.set_ylim(8.5, 12.5)
    ax_main.grid(True, alpha=0.3)
    ax_main.legend(loc='lower right', fontsize=10, title=RUN_LABELS.get('Default'), title_fontsize=10)

    ax_ratio.axhline(1.0, color='0.4', ls=':', lw=1)
    ax_ratio.set_xlabel(r'$\log_{10}\,M_\mathrm{halo}\ [M_\odot/h]$')
    ax_ratio.set_xlim(11.0, 15.3)
    ax_ratio.set_ylim(0.25, 4.0)
    ax_ratio.set_yscale('log')
    ax_ratio.set_yticks([0.3, 0.5, 1.0, 2.0, 3.0])
    ax_ratio.set_yticklabels(['0.3', '0.5', '1', '2', '3'])
    ax_ratio.grid(True, alpha=0.3, which='both')
    if j == 0:
        ax_ratio.set_ylabel(r'$M_\star\,/\,M_\star^\mathrm{default}$')

axes[0, 0].legend(loc='upper left', fontsize=10, frameon=False)
fig.suptitle('FOF-central stellar-to-halo mass relation across $\\tau_0$ variants', y=0.995)
plt.show()

## Figure 2 — Per-halo BCG stellar-mass distribution at fixed $M_\mathrm{halo}$

The median curves show what happens to a *typical* BCG. The full distribution is more diagnostic at the high-mass end because $\tau_0$ should reshape both the location and the width of the BCG mass distribution: $\tau_0=0$ should harden the high-$M_\star$ tail (every halo's BCG accretes everything), while $\tau_0=\infty$ should suppress it.

We pick a high-mass bin and a Milky-Way-mass bin and pool BCGs across all 16 ivols of each run.

In [ ]:
snapshot = 'iz271'  # z=0
z_val = SNAPSHOTS[snapshot][1]
halo_bin_edges = [(11.7, 12.3, 'Milky-Way scale\n$\\log_{10}M_h \\in [11.7, 12.3]$'),
                  (13.7, 14.3, 'Group scale\n$\\log_{10}M_h \\in [13.7, 14.3]$')]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
ms_bins = np.arange(8.5, 12.51, 0.1)

for ax, (lo, hi, title) in zip(axes, halo_bin_edges):
    for run_label, run_path in RUNS.items():
        log_ms_pool = []
        for iv in ivols:
            data = load_galaxy_fields(run_path / snapshot, iv)
            if data is None:
                continue
            mask = fof_central_mask(data) & (data['mstar'] > 0)
            log_mh = np.log10(data['mhhalo'][mask])
            log_ms = np.log10(data['mstar'][mask])
            in_bin = (log_mh >= lo) & (log_mh < hi)
            log_ms_pool.append(log_ms[in_bin])
        if not log_ms_pool:
            continue
        log_ms_all = np.concatenate(log_ms_pool)
        st = style_for(run_label)
        ax.hist(
            log_ms_all,
            bins=ms_bins,
            density=True,
            histtype='step',
            lw=2.2,
            color=st['color'],
            label=f"{st['label']}  (N={log_ms_all.size:,})",
        )
        ax.axvline(np.median(log_ms_all), color=st['color'], lw=1.0, ls='--', alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel(r'$\log_{10}\,M_{\star,\,\mathrm{cen}}\ [M_\odot/h]$')
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('PDF')
fig.suptitle(f'BCG stellar-mass distribution at fixed $M_\\mathrm{{halo}}$  (z = {z_val:.1f})', y=1.02)
plt.tight_layout()
plt.show()

## Figure 3 — Logarithmic scatter $\sigma(\log M_\star\,|\,M_h)$

$\tau_0$ should not only shift the mean SHMR but also broaden / narrow the per-halo scatter, because merger histories differ from halo to halo. We approximate the scatter as $(p_{84}-p_{16})/2$ in dex (a robust 1-σ-like estimator), averaged across ivols.

A signature of $\tau_0$ mattering at the high-mass end is the scatter behaviour at $\log_{10}M_h \gtrsim 13$.

In [ ]:
fig, axes = plt.subplots(1, len(SNAPSHOTS), figsize=(11, 4.5), sharey=True, sharex=True)
if len(SNAPSHOTS) == 1:
    axes = [axes]

for j, snapshot in enumerate(SNAPSHOTS):
    z_val = SNAPSHOTS[snapshot][1]
    ax = axes[j]
    for run_label in RUNS:
        s = stacked[snapshot][run_label]
        if s['n_used'] == 0:
            continue
        scatter = 0.5 * (s['p84']['mean'] - s['p16']['mean'])
        n_bin_mean = s['counts']['mean']
        ok = np.isfinite(scatter) & (n_bin_mean >= min_count)
        st = style_for(run_label)
        ax.plot(halo_centers[ok], scatter[ok], '-', lw=2.2, **st)
    ax.set_title(f'{snapshot}  (z = {z_val:.1f})')
    ax.set_xlabel(r'$\log_{10}\,M_\mathrm{halo}\ [M_\odot/h]$')
    ax.grid(True, alpha=0.3)
    if j == 0:
        ax.set_ylabel(r'$\sigma(\log_{10} M_\star\,|\,M_h)$  [dex]')
    ax.legend(fontsize=10)
fig.suptitle(r'Per-halo BCG scatter as a function of $M_\mathrm{halo}$', y=1.02)
plt.tight_layout()
plt.show()

## Quantitative summary

Numerical bracket on the $\tau_0$ effect at three diagnostic halo masses: Milky-Way ($\log_{10}M_h \sim 12$), group ($\sim 13$), and cluster ($\sim 14$).

In [ ]:
import pandas as pd

rows = []
for snapshot in SNAPSHOTS:
    z_val = SNAPSHOTS[snapshot][1]
    default_med = stacked[snapshot]['Default']['median']['mean']
    for log_mh_target in (12.0, 13.0, 14.0):
        i = int(np.argmin(np.abs(halo_centers - log_mh_target)))
        for run_label in RUNS:
            s = stacked[snapshot][run_label]
            if s['n_used'] == 0:
                continue
            med = s['median']['mean'][i]
            scatter = 0.5 * (s['p84']['mean'][i] - s['p16']['mean'][i])
            ratio = 10 ** (med - default_med[i]) if np.isfinite(default_med[i]) else np.nan
            rows.append({
                'snapshot': snapshot,
                'z': z_val,
                'log10_Mhalo': log_mh_target,
                'run': run_label,
                'median_logMstar': round(med, 3),
                'sigma_logMstar': round(scatter, 3),
                'Mstar/Mstar_default': round(ratio, 3),
                'n_BCGs_per_ivol': int(s['counts']['mean'][i]),
            })
df = pd.DataFrame(rows)
df

## Interpretation

Read the top-row panels and the ratio panels of Figure 1 together:

* If the three curves diverge at large $M_h$, $\tau_0$ matters in the high-mass regime.
* The $\tau_0=0$ curve should sit *above* the Default curve at large $M_h$ (BCG accretes its satellites), and the $\tau_0=\infty$ curve should sit *below* it. The amplitude of this split is the headline number for the paper.
* If the curves are close at $\log_{10}M_h \lesssim 12$ but visibly split at $\log_{10}M_h \gtrsim 13$, that is direct evidence that the dynamical-friction sink term dominates the BCG mass budget in groups and clusters.
* The redshift comparison (z=0 vs z=0.5) tells you how much of the bracket is built up over the last ~5 Gyr of cosmic time.
* Figure 2 panels show whether the effect is in the *median* or the *tail* — a $\tau_0=0$ run should harden the high-$M_\star$ tail.
* Figure 3 says how much $\tau_0$ broadens or narrows the per-halo scatter — a key observable for hydrodynamical/SAM comparison.